# EDA walkthrough

Companion to the course pages *EDA basics* and *How to: explore a dataset before analyzing it*.

By default this runs on a **made-up lipid-clinic cohort** (Mia's LDLR variant study) that has some problems planted in it. Try to find them before reading the last cell. To use your own data, set `DATA_PATH` and `TARGET` below.

The goal of exploring is to understand the data and generate questions, not to confirm anything. Needs `numpy`, `pandas`, `matplotlib`, `seaborn`.

In [ ]:
# ---- Parameters: change these and re-run ----
DATA_PATH = None           # e.g. "my_data.csv"; the made-up demo dataset is used if None
TARGET = "ldl_mmol"       # the outcome column you care about (None to skip the outcome-focused plots)
SPLIT_BY = "statin"        # a categorical column to split subgroup plots by (None to skip)
DATE_COLUMNS = ["sample_date"]   # columns to parse as dates (empty list if none)
MAX_CATEGORIES = 12        # columns with at most this many distinct values are treated as categories
SEED = 0

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns


def make_demo(n=300):
    """Made-up lipid-clinic cohort with some problems planted in it (see the last cell)."""
    rng = np.random.default_rng(SEED)
    clinic = rng.choice([f"C{i}" for i in range(1, 11)], n)
    statin = rng.choice(["no", "yes"], n, p=[0.6, 0.4])
    carrier_status = rng.choice(["non-carrier", "carrier"], n)
    age = rng.normal(50, 14, n).clip(18)
    days = rng.integers(0, 120, n)
    boost = np.where((carrier_status == "carrier") & (statin == "no"), 0.9, 0.0)
    y = 4.2 + boost - ((age - 55) / 25) ** 2 - 0.004 * days + rng.normal(0, 0.5, n)
    df = pd.DataFrame({
        "clinic": clinic, "statin": statin, "carrier_status": carrier_status, "age": age.round(0),
        "sample_date": pd.Timestamp("2024-06-01") + pd.to_timedelta(days, unit="D"),
        "ldl_mmol": y.round(1),
    })
    missing = (rng.random(n) < 0.04) | ((df["age"] < 32) & (rng.random(n) < 0.6)) | ((df["clinic"] == "C7") & (rng.random(n) < 0.5))
    df.loc[missing, "ldl_mmol"] = np.nan
    df.loc[17, "ldl_mmol"] = -3.2       # a typo
    df.loc[88, "ldl_mmol"] = 39.0       # a decimal slip
    return df


df = pd.read_csv(DATA_PATH, parse_dates=DATE_COLUMNS) if DATA_PATH else make_demo()

datetime_cols = df.select_dtypes("datetime").columns.tolist()
categorical = [c for c in df.columns if c not in datetime_cols
               and (not pd.api.types.is_numeric_dtype(df[c]) or df[c].nunique() <= MAX_CATEGORIES)]
numeric = [c for c in df.columns if c not in datetime_cols + categorical]
print(f"{df.shape[0]} rows x {df.shape[1]} columns")
print("numeric:", numeric, "| categorical:", categorical, "| dates:", datetime_cols)
df.head()

## 1. Data quality: ranges, duplicates, missingness

Read the `min` and `max` columns first: is anything impossible or suspicious? Then ask who is missing, because missing values that depend on other variables can't just be dropped without thought.

In [ ]:
print(df.describe(include="number").T[["count", "mean", "std", "min", "max"]].round(2))
print("\nduplicate rows:", df.duplicated().sum())

missing = df.isna().mean().sort_values(ascending=False)
print("\nshare missing:\n", missing[missing > 0].round(3))
fig, ax = plt.subplots(figsize=(6, 2 + 0.3 * len(missing)))
missing.plot.barh(ax=ax)
ax.set_xlabel("fraction missing")
plt.show()

if TARGET:
    print(f"\nAverage of the numeric columns, by whether {TARGET} is missing:")
    print(df.groupby(df[TARGET].isna())[numeric].mean().round(1))
    for cat in categorical:
        share = df.groupby(cat)[TARGET].apply(lambda s: s.isna().mean()).round(2)
        print(f"\nShare of {TARGET} missing, by {cat}:\n", share.to_string())

## 2. One variable at a time

Look for impossible values, isolated far-away points, pile-ups, and two humps.

In [ ]:
df[numeric].hist(bins=30, figsize=(10, 3 * ((len(numeric) + 1) // 2)), layout=((len(numeric) + 1) // 2, 2))
plt.tight_layout()
plt.show()

fig, axes = plt.subplots(1, max(len(categorical), 1), figsize=(4 * max(len(categorical), 1), 3))
for ax, cat in zip(np.atleast_1d(axes), categorical):
    df[cat].value_counts(dropna=False).plot.bar(ax=ax, title=cat)
plt.tight_layout()
plt.show()

## 3. Pairs of variables

Scatterplots reveal curves, outliers and clusters that a correlation coefficient hides. Extreme values stretch the axes, so you may want to set them aside (for plotting only) once you have spotted them: set `PLOT_RANGE` to your outcome's plausible range.

In [ ]:
PLOT_RANGE = (0, 10)   # plausible range of the outcome, for plotting only; None to plot everything

plot_df = df
if TARGET and PLOT_RANGE:
    plot_df = df[df[TARGET].between(*PLOT_RANGE) | df[TARGET].isna()]
    print(f"Plotting {len(plot_df)} of {len(df)} rows (outcome outside {PLOT_RANGE} left out of the plots only).")

sns.pairplot(plot_df, vars=numeric[:6], hue=SPLIT_BY, corner=True)
plt.show()

## 4. The outcome by group, and subgroup structure

Boxes summarize; the points show how much data is behind each box. Colouring by a second category (`SPLIT_BY`) shows whether the pattern is the same in every subgroup.

In [ ]:
if TARGET:
    for cat in categorical:
        if cat == TARGET:
            continue
        fig, ax = plt.subplots(figsize=(7, 4))
        hue = SPLIT_BY if SPLIT_BY and SPLIT_BY != cat else None
        sns.boxplot(data=plot_df, x=cat, y=TARGET, hue=hue, ax=ax)
        if hue is None:
            sns.stripplot(data=plot_df, x=cat, y=TARGET, color="black", alpha=0.3, ax=ax)
        plt.show()

    for col in numeric:
        if col == TARGET:
            continue
        fig, ax = plt.subplots(figsize=(6, 4))
        sns.scatterplot(data=plot_df, x=col, y=TARGET, hue=SPLIT_BY, alpha=0.6, ax=ax)
        plt.show()

## 5. Trends over time

In [ ]:
if TARGET:
    for date_col in datetime_cols:
        ordered = plot_df.sort_values(date_col)
        fig, ax = plt.subplots(figsize=(8, 3))
        ax.plot(ordered[date_col], ordered[TARGET], ".", alpha=0.5)
        ax.plot(ordered[date_col], ordered[TARGET].rolling(30, min_periods=10).mean(), color="red", label="rolling mean")
        ax.set_title(f"{TARGET} over {date_col}")
        ax.legend()
        plt.show()

## What to do with what you saw

Write down what you noticed *as questions*. Anything you found by looking is a hypothesis to check on new data, not a result. Fix clear data errors at the source if you can; otherwise exclude them and say so, with the number of rows affected.

---

**Spoiler: what is planted in the demo data**

- One impossible LDL value (negative) and one 10x value that looks like a decimal slip: data errors, visible in the min/max table and the histogram.
- Missing LDL values that are not random: more common in young patients and in clinic C7. Dropping those rows would quietly remove the young patients from the analysis.
- The variant's LDL-raising effect exists only in patients not taking statins (statins mask it): it is invisible in the overall comparison and obvious once you split by statin use. You found it by looking, so it is a lead to test on new data, not a confirmed effect.
- A curved age relationship (LDL peaks at middle age): a straight-line fit would miss it.
- A downward drift with sample date, which could be a change in the assay or in clinical practice.

Next steps depend on your goal: see the course pages on missing data and on prediction vs. causation.